# Phase 3: MuRIL + IndicBERT fine-tuning (Hindi)

Owner: Shreyank Sthavaramath. Plan: [docs/01-roadmap.md](../docs/01-roadmap.md), Phase 3.

Fine-tune **MuRIL** (primary) and **IndicBERT** (comparison) for Hindi NER with
HuggingFace `AutoModelForTokenClassification` + `Trainer`. The subword
label-alignment step is the #1 NER bug (see
[docs/05-pitfalls.md](../docs/05-pitfalls.md)); it gets its own section and a
self-test below before any model is trained on it.

**The training size is found from the data, not fixed up front.** Train at an
increasing sequence of subset sizes, score Hindi gold F1 after each, log every
point as its own row in the results sheet, and stop adding data once the
F1-vs-size curve flattens. That size becomes the working subset for the rest
of the project.

**Gate:** both models scored on Hindi gold; MuRIL clearly beats the
BiLSTM-CRF baseline (**0.8038**, Phase 2); the scaling curve is logged and a
working subset size is chosen from it. *If MuRIL does not beat the baseline,
suspect the label alignment first.*

**How to run**
1. Run once locally or on Colab with `DEBUG = True`: ~80 sentences, 1 epoch,
   CPU is fine, finishes in a few minutes. This only checks the pipeline
   end-to-end (label alignment, training loop, eval, logging) -- the printed
   scores are meaningless at that size and are never logged as a real result.
2. For the real scaling curve, open this notebook on **Kaggle** (a GPU
   runtime; see [docs/04-compute.md](../docs/04-compute.md)) with
   `DEBUG = False`, and run the "Scaling curve" section. Each step
   checkpoints to Kaggle working storage / Drive so a disconnect only costs
   the current step.
3. Look at the plotted curve, fill in `CHOSEN_SUBSET_SIZE` in the "Choose the
   working subset size" cell, then run the IndicBERT comparison cell at that
   size.
4. Check the gate with `uv run python scripts/phase3_gate.py` (from the repo
   root, locally -- it only reads `results/results.csv` and this notebook,
   no GPU needed).

In [1]:
# Pin versions (docs/04-compute.md). Colab/Kaggle ship their own torch;
# transformers/datasets/accelerate/seqeval/sentencepiece are pinned here so
# runs are reproducible across machines.
%pip install -q transformers==5.22.1 datasets==5.1.0 accelerate==1.15.0 seqeval==1.2.2 sentencepiece==0.2.2

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/iPriyadarshi/NaamRekha.git"

# Local: the notebook runs from notebooks/, so the repo root is "..".
# Colab/Kaggle: clone the repo next to the notebook.
if os.path.isdir("../src"):
    REPO_ROOT = os.path.abspath("..")
elif os.path.isdir("src"):
    REPO_ROOT = os.path.abspath(".")
else:
    REPO_ROOT = os.path.abspath("NaamRekha")
    if not os.path.isdir(REPO_ROOT):
        # If the repo is private, use https://<token>@github.com/... or upload src/ manually.
        url = REPO_URL
        token = None
        try:
            from google.colab import userdata
            token = userdata.get("GH_TOKEN")      # Colab Secret, if available
        except Exception:
            pass
        if not token:
            from getpass import getpass
            token = getpass("GitHub token (Enter to skip if repo is public): ")
        if token:
            url = REPO_URL.replace("https://", f"https://{token}@")
        subprocess.run(["git", "clone", "--depth", "1", url, REPO_ROOT], check=True)
sys.path.insert(0, REPO_ROOT)

from src import LABELS, evaluate, id2label, label2id, load_split, log_result, tag_distribution

print("repo:", REPO_ROOT)
print("labels:", LABELS)

GitHub token (Enter to skip if repo is public):  ········


Cloning into '/kaggle/working/NaamRekha'...


repo: /kaggle/working/NaamRekha
labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


## Config

Every knob is in this one cell. Each `train_run()` call records its settings
in the results-row `notes` automatically.

In [ ]:
SEED = 42
LANG = "hi"

DEBUG = False   # True: ~80 sentences, 1 epoch (pipeline check). False: real runs on Kaggle.

# Kaggle's dual-T4 sessions otherwise wrap the model in DataParallel, silently
# doubling the effective batch size below. Force single-GPU so BATCH_SIZE is
# the real one (must be set before torch is imported/initializes CUDA).
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# MuRIL is the primary model (roadmap). IndicBERT is the comparison model.
MODELS = {
    "MuRIL": "google/muril-base-cased",
    "IndicBERT": "ai4bharat/indic-bert",
}

MAX_LENGTH = 128      # subword tokens per sentence (docs/04-compute.md: NER sentences are short)
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
LR = 2e-5
EPOCHS = 3

# Scaling ladder (roadmap: "10k, then roughly triple each step"). 900k was
# never completed in a Kaggle session and isn't needed (the curve already
# flattens by 100k-300k), so it's dropped from the ladder.
SUBSET_SIZES = [10_000, 30_000, 100_000, 300_000]

if DEBUG:
    SUBSET_SIZES = [80]
    EPOCHS = 1
    BATCH_SIZE = 8


In [7]:
def pick_storage() -> str:
    # Persistent storage for checkpoints: Kaggle working dir, Google Drive, or local checkpoints/.
    if os.path.isdir("/kaggle/working"):
        return "/kaggle/working/naamrekha"
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        return "/content/drive/MyDrive/NaamRekha"
    except ImportError:
        return os.path.join(REPO_ROOT, "checkpoints")


STORAGE = pick_storage()
os.makedirs(os.path.join(STORAGE, "phase3"), exist_ok=True)
print("checkpoints ->", os.path.join(STORAGE, "phase3"))

checkpoints -> /kaggle/working/naamrekha/phase3


In [ ]:
import random

import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device, "| GPU count:", torch.cuda.device_count())
if device.type == "cpu":
    print("WARNING: no GPU. Fine for DEBUG, too slow for the real scaling runs -- use Kaggle.")


## Data

Train on **silver**, evaluate on the full **gold** test set, per
[docs/03-conventions.md](../docs/03-conventions.md). `make_subset` draws a
seeded random subset (same recipe as Phase 2) and `entity_report` checks it
is not mostly `O`, per the subset rule in that doc.

In [9]:
train_full = load_split(LANG, "train")
val = load_split(LANG, "val")
test = load_split(LANG, "test")
print(f"train {len(train_full):,} (full silver) | val {len(val):,} | test {len(test):,} (gold)")


def make_subset(rows, n, seed=SEED):
    n = min(n, len(rows))
    idx = sorted(random.Random(seed).sample(range(len(rows)), n))
    return [rows[i] for i in idx]


def entity_report(rows, name):
    counts = tag_distribution(rows)
    total = sum(counts.values())
    ents = {t: counts.get(f"B-{t}", 0) for t in ("PER", "ORG", "LOC")}
    with_ent = sum(any(t != "O" for t in tags) for _, tags in rows)
    print(f"{name}: entities {ents} | O = {counts['O'] / total:.1%} of tokens | "
          f"{with_ent / len(rows):.1%} of sentences have >=1 entity")


entity_report(test, "gold test (fixed, never subset)")

data/hi_IndicNER_v1.0.zip: reconstructing file:   0%|          |  0.00B / 82.3MB            

data/hi_IndicNER_v1.0.zip: downloading bytes:           |  0.00B            

train 985,787 (full silver) | val 13,460 | test 864 (gold)
gold test (fixed, never subset): entities {'PER': 788, 'ORG': 521, 'LOC': 613} | O = 83.0% of tokens | 86.0% of sentences have >=1 entity


## Subword label alignment

The #1 NER bug with BERT-family models: each tokenizer splits a word into
one or more subword pieces, but we have one label per **word**. The rule
(docs/05-pitfalls.md): label only the first subword of each word with that
word's tag; every continuation subword and every special token (`[CLS]`,
`[SEP]`, padding) gets `-100` so the loss ignores it. `word_ids()` is what
makes this mapping possible.

A self-test below checks this on a real training sentence *before* any
model is trained on it.

In [10]:
from datasets import Dataset


def tokenize_and_align_labels(batch, tokenizer, max_length=MAX_LENGTH):
    enc = tokenizer(batch["words"], is_split_into_words=True, truncation=True, max_length=max_length)
    all_labels = []
    for i, tags in enumerate(batch["tags"]):
        word_ids = enc.word_ids(batch_index=i)
        prev = None
        label_ids = []
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)          # [CLS] / [SEP] / padding
            elif wid != prev:
                label_ids.append(label2id[tags[wid]])   # first subword of this word
            else:
                label_ids.append(-100)          # continuation subword
            prev = wid
        all_labels.append(label_ids)
    enc["labels"] = all_labels
    return enc


def to_hf_dataset(rows, tokenizer, max_length=MAX_LENGTH):
    ds = Dataset.from_dict({"words": [r[0] for r in rows], "tags": [r[1] for r in rows]})
    return ds.map(lambda b: tokenize_and_align_labels(b, tokenizer, max_length),
                  batched=True, remove_columns=["words", "tags"])

In [ ]:
from transformers import AutoTokenizer

for _model_key, _checkpoint in MODELS.items():
    _tok = AutoTokenizer.from_pretrained(_checkpoint)
    _sample_words, _sample_tags = train_full[0]
    _check_ds = to_hf_dataset([train_full[0]], _tok)
    _enc_check = _tok(_sample_words, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH)
    _word_ids_check = _enc_check.word_ids()
    _labels_check = _check_ds[0]["labels"]

    assert len(_labels_check) == len(_word_ids_check)
    _seen = set()
    for _pos, _wid in enumerate(_word_ids_check):
        if _wid is None:
            assert _labels_check[_pos] == -100, "special token must be -100"
        elif _wid not in _seen:
            assert _labels_check[_pos] == label2id[_sample_tags[_wid]], "first subword must carry the word's tag"
            _seen.add(_wid)
        else:
            assert _labels_check[_pos] == -100, "continuation subword must be -100"

    print(f"label-alignment self-test ({_model_key}, {_checkpoint}): PASSED "
          f"({len(_sample_words)} words -> {len(_word_ids_check)} subword positions)")

# kept for cell 13's truncation count, which checks against MuRIL's tokenizer specifically
_test_tokenizer = AutoTokenizer.from_pretrained(MODELS["MuRIL"])


## Prediction and scoring

Same evaluator as every other phase (`src.eval.evaluate`, entity-level
`seqeval` micro-F1). Predictions are mapped back from subwords to words the
same way the labels were built: only the first-subword position of each
word is read; `-100`/continuation positions are skipped.

In [12]:
def predict_tags(model, tokenizer, rows, max_length=MAX_LENGTH, batch_size=EVAL_BATCH_SIZE):
    model.eval()
    dev = next(model.parameters()).device
    y_true, y_pred = [], []
    for i in range(0, len(rows), batch_size):
        chunk = rows[i:i + batch_size]
        words_batch = [r[0] for r in chunk]
        tags_batch = [r[1] for r in chunk]
        enc = tokenizer(words_batch, is_split_into_words=True, truncation=True, max_length=max_length,
                         padding=True, return_tensors="pt").to(dev)
        with torch.no_grad():
            logits = model(**enc).logits
        preds = logits.argmax(-1).cpu().numpy()
        for b, tags in enumerate(tags_batch):
            word_ids = enc.word_ids(batch_index=b)
            pred_tags, true_tags, prev = [], [], None
            for pos, wid in enumerate(word_ids):
                if wid is None or wid == prev:
                    continue
                pred_tags.append(id2label[preds[b][pos]])
                true_tags.append(tags[wid])
                prev = wid
            y_pred.append(pred_tags)
            y_true.append(true_tags)
    return y_true, y_pred


_long = sum(1 for words, _ in test if len(_test_tokenizer(words, is_split_into_words=True)["input_ids"]) > MAX_LENGTH)
print(f"gold test sentences with > {MAX_LENGTH} subword tokens (tail truncated at eval): {_long} / {len(test)}")

gold test sentences with > 128 subword tokens (tail truncated at eval): 0 / 864


## Reusable training run

One function, called once per `(model, subset_size)` point on the scaling
curve. It samples the subset, checks entity density, fine-tunes with
HuggingFace `Trainer` (checkpointing every epoch so a Kaggle disconnect only
costs the current step), scores on the full gold test set, and logs one
`results.csv` row -- exactly the Phase 2 contract, just driven from code
instead of copy-pasted per run.

In [ ]:
import math

from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)


def train_run(model_key, subset_size, epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, seed=SEED, resume=True):
    checkpoint = MODELS[model_key]
    random.seed(seed)
    torch.manual_seed(seed)

    train = make_subset(train_full, subset_size, seed)
    entity_report(train, f"{model_key} train subset n={len(train)}")

    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForTokenClassification.from_pretrained(
        checkpoint, num_labels=len(label2id), id2label=id2label, label2id=label2id
    )
    train_ds = to_hf_dataset(train, tokenizer)
    collator = DataCollatorForTokenClassification(tokenizer)

    run_name = f"{model_key.lower()}_{LANG}_n{len(train)}_s{seed}" + ("_debug" if DEBUG else "")
    ckpt_dir = os.path.join(STORAGE, "phase3", run_name)
    has_ckpt = os.path.isdir(ckpt_dir) and any(f.startswith("checkpoint-") for f in os.listdir(ckpt_dir))

    # CUDA_VISIBLE_DEVICES=0 (Config cell) keeps this at exactly one GPU, so
    # batch_size below is the real effective batch size, not silently doubled
    # by DataParallel across Kaggle's two T4s.
    n_gpu = max(torch.cuda.device_count(), 1)
    total_steps = math.ceil(len(train_ds) / (batch_size * n_gpu)) * epochs
    warmup_steps = int(0.1 * total_steps)

    args = TrainingArguments(
        output_dir=ckpt_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        learning_rate=lr,
        warmup_steps=warmup_steps,
        fp16=torch.cuda.is_available(),
        save_strategy="epoch",
        save_total_limit=2,
        logging_steps=50,
        seed=seed,
        report_to=[],
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator,
                       processing_class=tokenizer)
    trainer.train(resume_from_checkpoint=has_ckpt and resume)
    trainer.save_model(ckpt_dir)

    y_true, y_pred = predict_tags(model, tokenizer, test)
    scores = evaluate(y_true, y_pred)

    notes = (f"seed={seed}; lr={lr}; bs={batch_size}; gpus={n_gpu}; warmup_steps={warmup_steps}; "
             f"epochs={epochs}; max_len={MAX_LENGTH}; checkpoint={checkpoint}"
             + ("; DEBUG run -- not a real result" if DEBUG else ""))
    log_result(model=model_key, train_lang=LANG, test_lang=LANG, subset_size=len(train),
                epochs=epochs, scores=scores, checkpoint_path=ckpt_dir, notes=notes)

    with open(os.path.join(REPO_ROOT, "results", "results.csv"), encoding="utf-8") as f:
        print(f.read().splitlines()[-1])
    return scores


## Pipeline check (`DEBUG = True`)

Runs the whole pipeline once on a tiny CPU-sized slice. The score is not
meaningful (too little data, 1 epoch) -- it is logged with `DEBUG` in
`notes` so `scripts/phase3_gate.py` and the scaling-curve plot both ignore
it. This is only proof that label alignment, training, prediction, scoring,
and logging all work end-to-end before spending Kaggle GPU hours on it.

In [11]:
if DEBUG:
    debug_scores = train_run("MuRIL", SUBSET_SIZES[0])
    print("DEBUG scores (meaningless at this size):", debug_scores)

## Scaling curve (run on Kaggle, `DEBUG = False`)

Trains MuRIL at each size in `SUBSET_SIZES`, largest last, so an early
disconnect still leaves the smaller points logged. Each call is independent
and resumable (`train_run(..., resume=True)` continues from the last epoch
checkpoint for that size if the kernel restarts).

In [ ]:
import shutil

if not DEBUG:
    for _n in SUBSET_SIZES:
        train_run("MuRIL", _n)
        ckpt_dir = os.path.join(STORAGE, "phase3", f"muril_hi_n{_n}_s{SEED}")
        shutil.rmtree(ckpt_dir, ignore_errors=True)
        print(f"cleaned up checkpoint for n={_n}")


MuRIL train subset n=10000: entities {'PER': 7546, 'ORG': 7079, 'LOC': 7435} | O = 82.2% of tokens | 90.6% of sentences have >=1 entity


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ig

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

... (42 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MuRIL,hi,hi,300000,3,0.89079754601227,0.7517985611510791,0.8162291169451075,0.8287071767941986,/kaggle/working/naamrekha/phase3/muril_hi_n300000_s42,seed=42; lr=2e-05; bs=32; epochs=3; max_len=128; checkpoint=google/muril-base-cased
cleaned up checkpoint for n=300000
MuRIL train subset n=900000: entities {'PER': 699987, 'ORG': 626388, 'LOC': 668037} | O = 82.2% of tokens | 90.4% of sentences have >=1 entity


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

## Plot: F1 vs. training size

Reads the logged (non-`DEBUG`) MuRIL `hi` -> `hi` rows straight from
`results/results.csv`, so the plot is always built from what is actually
logged, not from in-memory state.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

results_df = pd.read_csv(os.path.join(REPO_ROOT, "results", "results.csv"))
curve = results_df[
    (results_df["model"] == "MuRIL")
    & (results_df["train_lang"] == LANG)
    & (results_df["test_lang"] == LANG)
    & (~results_df["notes"].str.contains("DEBUG", na=False))
].sort_values("subset_size")

if len(curve) == 0:
    print("No real (non-DEBUG) MuRIL rows logged yet -- run the scaling-curve cell on Kaggle first.")
else:
    baseline_f1 = results_df.loc[
        (results_df["model"] == "BiLSTM-CRF") & (~results_df["notes"].str.contains("DEBUG", na=False)),
        "F1_overall",
    ].max()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(curve["subset_size"], curve["F1_overall"], marker="o", label="MuRIL (hi gold F1)")
    ax.axhline(baseline_f1, color="gray", linestyle="--", label=f"BiLSTM-CRF baseline ({baseline_f1:.4f})")
    ax.set_xscale("log")
    ax.set_xlabel("training subset size (sentences, log scale)")
    ax.set_ylabel("Hindi gold entity F1 (overall)")
    ax.set_title("Phase 3: MuRIL scaling curve")
    ax.legend()
    fig.tight_layout()
    fig_path = os.path.join(REPO_ROOT, "results", "phase3_scaling_curve.png")
    fig.savefig(fig_path, dpi=150)
    print("saved", fig_path)
    print(curve[["subset_size", "F1_overall"]].to_string(index=False))

## Choose the working subset size

Fill this in by eye from the plot above: the smallest size where the next
step's F1 gain is no longer "clear" (the roadmap's stopping rule -- grow
while the slope is steep, stop once it flattens). If the curve is still
climbing steeply at the largest size a Kaggle session allows, say so in
[docs/phase3/README.md](../docs/phase3/README.md) rather than forcing a
plateau that isn't there.

In [18]:
CHOSEN_SUBSET_SIZE = 100_000  # set from the plot above, e.g. 300_000
if CHOSEN_SUBSET_SIZE is not None:
    print("working subset size:", CHOSEN_SUBSET_SIZE)

working subset size: 100000


## IndicBERT comparison (run on Kaggle, after choosing the size above)

One run at `CHOSEN_SUBSET_SIZE`, same harness, so MuRIL and IndicBERT are
compared at the same training size.

In [ ]:
if not DEBUG and CHOSEN_SUBSET_SIZE is not None:
    indicbert_scores = train_run("IndicBERT", CHOSEN_SUBSET_SIZE)
    print("IndicBERT scores:", indicbert_scores)

    from transformers import AutoModelForTokenClassification, AutoTokenizer
    ckpt_dir = os.path.join(STORAGE, "phase3", f"indicbert_{LANG}_n{CHOSEN_SUBSET_SIZE}_s{SEED}")
    model = AutoModelForTokenClassification.from_pretrained(ckpt_dir)
    tokenizer = AutoTokenizer.from_pretrained(ckpt_dir)
    repo_id = "shreyanks-1/indicbert-hindi-ner-100k"
    model.push_to_hub(repo_id)
    tokenizer.push_to_hub(repo_id)
    print(f"Pushed to https://huggingface.co/{repo_id}")


IndicBERT train subset n=100000: entities {'PER': 78044, 'ORG': 70041, 'LOC': 74368} | O = 82.0% of tokens | 90.6% of sentences have >=1 entity


config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  135MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/23 [00:00<?, ?it/s]

... (19 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

Pushed to https://huggingface.co/shreyanks-1/indicbert-hindi-ner-100k


## Re-run MuRIL at the chosen size, keeping the checkpoint this time

The scaling-curve run at this size had its checkpoint deleted (disk quota cleanup).
This re-run uses the exact same config/seed and pushes the resulting checkpoint to
the Hugging Face Hub for a durable Phase 4 handoff.


In [ ]:
if not DEBUG and CHOSEN_SUBSET_SIZE is not None:
    muril_scores = train_run("MuRIL", CHOSEN_SUBSET_SIZE)
    print("MuRIL scores:", muril_scores)

    from transformers import AutoModelForTokenClassification, AutoTokenizer
    ckpt_dir = os.path.join(STORAGE, "phase3", f"muril_{LANG}_n{CHOSEN_SUBSET_SIZE}_s{SEED}")
    model = AutoModelForTokenClassification.from_pretrained(ckpt_dir)
    tokenizer = AutoTokenizer.from_pretrained(ckpt_dir)
    repo_id = "shreyanks-1/muril-hindi-ner-100k"
    model.push_to_hub(repo_id)
    tokenizer.push_to_hub(repo_id)
    print(f"Pushed to https://huggingface.co/{repo_id}")


MuRIL train subset n=100000: entities {'PER': 78044, 'ORG': 70041, 'LOC': 74368} | O = 82.0% of tokens | 90.6% of sentences have >=1 entity


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ig

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

... (3 outputs left out)

Step,Training Loss
50,3.645186
100,3.456445
150,3.368278
200,3.271963
250,3.170920
300,3.059839
350,2.948350
400,2.809964
450,2.686896
500,2.541703


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MuRIL,hi,hi,100000,3,0.904143475572047,0.7416741674167416,0.8223107569721115,0.8330404217926186,/kaggle/working/naamrekha/phase3/muril_hi_n100000_s42,seed=42; lr=2e-05; bs=32; epochs=3; max_len=128; checkpoint=google/muril-base-cased
MuRIL scores: {'F1_overall': 0.8330404217926186, 'F1_PER': 0.904143475572047, 'F1_ORG': 0.7416741674167416, 'F1_LOC': 0.8223107569721115}


## Gate

Run from the repo root (locally -- this only reads `results/results.csv`
and this notebook, no GPU needed):

```
uv run python scripts/phase3_gate.py
```

It checks: a scaling curve of at least 3 subset sizes logged for MuRIL, the
best MuRIL F1 clearly beating the Phase 2 baseline (0.8038), an IndicBERT
comparison row, every row traceable to this notebook's own printed output,
and [docs/phase3/README.md](../docs/phase3/README.md) reporting the chosen
subset size and the winning F1.